# State Data

## Processing the employment data

Using state_M2024_dl from the OEWS to get the employment per state at the occupation level

Filter on:
- AREA_TYPE: [2,3] 
- O_GROUP: Use `detailed`

Features to keep
- PRIM_STATE: First two letters of the state (common codification)
- AREA: State code in FIPS
- AREA_TITLE: Name of the state
- OCC_CODE: SOC code, !! Not the same as ONET!!
- TOT_EMP: Number of employees for the occ

In [ ]:
import pandas as pd
import numpy as np

#Anthropic penetration data
anthropic_penetration_file_path = "https://huggingface.co/datasets/Anthropic/EconomicIndex/resolve/main/labor_market_impacts/job_exposure.csv"
penetration_df = pd.read_csv(anthropic_penetration_file_path)
penetration_df = penetration_df[["occ_code", "observed_exposure"]].copy()
penetration_df.rename(columns={"occ_code": "OCC_CODE", "observed_exposure": "ai_penetration"}, inplace=True)

In [155]:
# Load
OEWS_state_df = pd.read_csv("./data/OEWS/state_M2024_dl.csv")
onet_mapping = pd.read_csv("./data/onet_code_to_id_mapping.csv")
exposure_df = pd.read_csv("./data/exposure_score/job_exposure_scores.csv")

OEWS_state_df = pd.merge(OEWS_state_df, penetration_df, on="OCC_CODE", how="left")
#


# Filter
OEWS_state_df = OEWS_state_df[
    OEWS_state_df["AREA_TYPE"].isin([2, 3]) &
    (OEWS_state_df["O_GROUP"] == "detailed")
].copy()

# Keep relevant features
OEWS_state_df = OEWS_state_df[
    ["AREA", "OCC_CODE", "TOT_EMP", "H_MEDIAN", "ai_penetration"]
].copy()

#AREa is in range 0 - 56, so we can safely convert to int8 to save memory
OEWS_state_df["AREA"] = OEWS_state_df["AREA"].astype(np.int8)
OEWS_state_df["TOT_EMP"] = pd.to_numeric(OEWS_state_df["TOT_EMP"], errors="coerce")

OEWS_state_df.replace({"TOT_EMP": {np.nan: 0}}, inplace=True)
OEWS_state_df.replace({"H_MEDIAN": {np.nan: 0}}, inplace=True)

OEWS_state_df.rename(columns={
    "AREA": "state_code",
    "OCC_CODE": "soc_code",
    "TOT_EMP": "weight",
    "H_MEDIAN": "h_median",
}, inplace=True)

exposure_df.rename(columns={"weighted_beta": "ai_ability"}, inplace=True)

# Add the ai_ability scores to the OEWS dataframe
OEWS_state_df = pd.merge(OEWS_state_df, exposure_df, on="soc_code")

# Add the soc id to the OEWS dataframe
OEWS_state_df = pd.merge(OEWS_state_df, onet_mapping, on = "soc_code", how="left")
OEWS_state_df.drop(columns=["soc_code"], inplace=True)

OEWS_state_df["ai_penetration"] = OEWS_state_df["ai_penetration"].fillna(0).round(3)
OEWS_state_df["ai_penetration"] = OEWS_state_df["ai_penetration"].round(3)
OEWS_state_df["ai_ability"] = OEWS_state_df["ai_ability"].round(3)
OEWS_state_df["weight"] = OEWS_state_df["weight"].astype(int)

OEWS_state_df["h_median"] = OEWS_state_df["h_median"].round(3)

OEWS_state_df.reset_index(drop=False, inplace=True)
OEWS_state_df.rename(columns={"index": "id"}, inplace=True)
OEWS_state_df["h_median"] = pd.to_numeric(OEWS_state_df["h_median"], errors="coerce")
OEWS_state_df = OEWS_state_df[["id", "state_code", "soc_id", "weight", "h_median", "ai_ability", "ai_penetration"]]

print(OEWS_state_df.head())
print(OEWS_state_df.shape)

OEWS_state_df.to_csv("./data/OEWS/processed_state_occ_OEWS.csv", index = False)


   id  state_code  soc_id  weight  h_median  ai_ability  ai_penetration
0   0           1       0     830     79.04       0.419           0.033
1   1           1       1       0     51.12       0.500           0.138
2   2           1       3      50     64.22       0.497           0.173
3   3           1       4       0     54.31       0.474           0.320
4   4           1       5       0     52.28       0.492           0.043
(33022, 7)


## Generating the GeoJson at the state level

In [ ]:
from pathlib import Path
import geopandas as gpd

# Paths
data_dir = Path("./data/maps")
state_zip = data_dir / "cb_2024_us_state_500k.zip"
out_geojson = data_dir / "us_states.geojson"

if not state_zip.exists():
    raise FileNotFoundError(
        f"Missing {state_zip}. Please place the Census state shapefile zip there."
    )

# Read local Census state boundaries
states_gdf = gpd.read_file(f"zip://{state_zip.resolve()}")

# Build 2-digit state FIPS key for joins with OEWS AREA
states_gdf["state_code"] = states_gdf["STATEFP"].astype(str).str.zfill(2)

# Keep only fields needed by frontend/join
states_out = states_gdf[["state_code", "NAME", "geometry"]].copy()

# Export GeoJSON
states_out.to_file(out_geojson, driver="GeoJSON")

print(f"Saved {out_geojson}")
print("Rows:", len(states_out))
print("Columns:", list(states_out.columns))
print(states_out[["state_code", "NAME"]].head())
